# M3: RGB-only self-supervised PoseNet + DepthNet

Use a private notebook in Suryansh's Kaggle account. This notebook clones the exact
reviewed `REVISION`; it does not use a local working tree or an earlier pilot
checkpoint. Attach the public `vaibhavkrishnabansal/vkitti2-subset` RGB input,
enable a GPU and Internet, and enable the notebook's read-only `GH_TOKEN` secret.
Do not paste the token into a cell. The canonical training config is unchanged
`configs/m3.yaml` (seed 42, 60 epochs, FP32, validation Sim(3) ATE selection).

For a fresh pilot version, set `RUN_PILOT=True`, `PILOT_STEPS=200`,
`RUN_FULL=False`, `EVALUATE=False`, `EVAL_DEPTH=False`, `DEPTH_ROOT=None`.
Verify the 18-sequence/8,642-frame check in the output and use **Save Version →
Save & Run All**. Inspect pilot losses, coverage, encoder updates, GPU memory,
and the three diagnostic PNGs. The mask rendering fix is in reviewed main.
The pilot uses a different selection key and scheduler and cannot seed full training.

For a fresh canonical version at the **same REVISION**, set `RUN_PILOT=False`,
`RUN_FULL=True`, `EVALUATE=True`, `EVAL_DEPTH=False`, `DEPTH_ROOT=None` and
Save & Run All. Keep the entire `/kaggle/working/checkpoints/m3` directory and
results before the session ends. To resume, restore best.pt, last.pt, history.csv
and summary.json to that directory before the training cell; it detects last.pt.
A failed background version must not be assumed to have published its outputs.

Depth is a later evaluation-only version after the separate PR merges. Pin that
merge SHA; attach the RGB input, your official VKITTI2 depth subset and your
own best.pt. Set `RUN_PILOT=False`, `RUN_FULL=False`, `EVALUATE=True`,
`EVAL_DEPTH=True`, `EVAL_CKPT` to the mounted checkpoint, `DEPTH_ROOT` to the
mounted directory containing Scene02/Scene20, and both dataset version fields.
The notebook checks paired depth frames and writes into a separate
`results/m3/depth_evaluation` directory. Keep depth GT out of training.
No code cell after the evaluation cell may fail and hide a saved version's output.


In [ ]:
from pathlib import Path
import os, re, subprocess, sys
import torch
assert torch.cuda.is_available(), "Enable a GPU accelerator before continuing"
print("torch", torch.__version__, "GPU", torch.cuda.get_device_name(0))
REPO = "VaibhavBansal05/Deep_Monocular_VO"
REVISION = "785e5ad933b3a0dd2ac96d940ebb981f0ab01709"  # reviewed main; retain for pilot and full run
CONFIG = "configs/m3.yaml"  # matched ablations: configs/m3_* yaml
RUN_PILOT = True  # prepared 200-step pilot; execute cell 7 only when ready
PILOT_STEPS = 200
RUN_FULL = False
EVALUATE = False  # enable only after freezing config and selecting validation checkpoint
EVAL_CKPT = None  # optional checkpoint path for evaluation-only; defaults to OUT/best.pt
EVAL_DEPTH = False  # requires the separately reviewed depth-evaluation integration
DEPTH_ROOT = None  # optional separate dataset root containing Scene02/Scene20 depth trees
RGB_DATASET_VERSION = "vaibhavkrishnabansal/vkitti2-subset/1"
DEPTH_DATASET_VERSION = None  # set to your private depth dataset/version for depth evaluation
NUM_WORKERS = 2
assert re.fullmatch(r"[0-9a-f]{40}", REVISION), "Set REVISION to your reviewed/pushed commit"


In [ ]:
from kaggle_secrets import UserSecretsClient
TOKEN = UserSecretsClient().get_secret("GH_TOKEN")
WORK = Path("/kaggle/working")
INPUT = "/kaggle/input"
CHECKPOINT_ROOT = WORK / "checkpoints"
RESULT_ROOT = WORK / "results"
# Persist/download /kaggle/working outputs before the session ends. Restore the
# checkpoint directory to this location before resuming in a new session.


In [ ]:
import tempfile
repo_path = WORK / "Deep_Monocular_VO"
# Askpass keeps the token out of the URL, saved cells and command error messages.
with tempfile.NamedTemporaryFile(mode="w", suffix=".sh", delete=False) as f:
    f.write('#!/bin/sh\ncase "$1" in *Username*) echo x-access-token ;; *) echo "$M3_GIT_TOKEN" ;; esac\n')
    askpass = f.name
os.chmod(askpass, 0o700)
environment = dict(os.environ, GIT_ASKPASS=askpass, GIT_TERMINAL_PROMPT="0", M3_GIT_TOKEN=TOKEN)
try:
    if not repo_path.exists():
        subprocess.run(["git", "clone", "--quiet", f"https://github.com/{REPO}.git", str(repo_path)], env=environment, check=True)
    else:
        assert not subprocess.check_output(["git", "status", "--porcelain"], cwd=repo_path, text=True).strip(), "Checkout contains changes; use a fresh runtime directory"
        subprocess.run(["git", "fetch", "origin"], cwd=repo_path, env=environment, check=True)
    subprocess.run(["git", "checkout", "--detach", REVISION], cwd=repo_path, check=True)
finally:
    os.unlink(askpass)
    environment.pop("M3_GIT_TOKEN", None)
    del TOKEN
assert subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo_path, text=True).strip() == REVISION
SRC = repo_path / "src"
os.chdir(SRC)
os.environ["PYTEST_DISABLE_PLUGIN_AUTOLOAD"] = "1"
# Preserve the platform's compatible torch/torchvision wheels.
PYTHON = sys.executable
subprocess.run([PYTHON, "-m", "pip", "install", "-e", ".[dev]", "--no-deps"], check=True)
def run(*args):
    subprocess.run([PYTHON, *args], check=True)
run("-m", "pytest", "-q", "-p", "no:cacheprovider")
# Feature detection keeps the standalone M3 branch usable without the resume PR.
HAS_CHECKPOINT_RESUME = subprocess.check_output(
    [PYTHON, "-c", "from mvo.common.train import engine; print(int(callable(getattr(engine, 'restore_rng', None))))"],
    text=True).strip() == "1"
print("Checkpoint resume integration:", "available" if HAS_CHECKPOINT_RESUME else "unavailable; fresh training supported")


In [ ]:
import glob, hashlib, shutil
roots = glob.glob(INPUT + "/**/vkitti2/Scene20/clone/extrinsic.txt", recursive=True)
tars = glob.glob(INPUT + "/**/vkitti2_subset.tar.gz", recursive=True)
if roots:
    DATA_ROOT = Path(roots[0]).parents[2]
elif tars:
    archive = Path(tars[0])
    assert archive.stat().st_size == 1514968986, "RGB archive is truncated"
    digest = hashlib.md5()
    with archive.open("rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            digest.update(block)
    assert digest.hexdigest() == "aca1947cc25aaa968c7d6c07d3c3ff01", "RGB checksum mismatch"
    data = WORK / "data"
    data.mkdir(exist_ok=True)
    subprocess.run(["tar", "xzf", str(archive), "-C", str(data)], check=True)
    DATA_ROOT = data / "vkitti2"
else:
    raise RuntimeError("Attach vkitti2_subset.tar.gz or the extracted vkitti2 tree")
from mvo.common.data import splits
sequence_count = total_frames = 0
for split in ("train", "val", "test", "robustness"):
    for scene, variation in splits.sequences(split):
        sequence = DATA_ROOT / scene / variation
        expected = splits.SCENE_LENGTHS[scene]
        frames = sorted((sequence / "frames/rgb/Camera_0").glob("rgb_*.jpg"))
        assert len(frames) == expected, (split, scene, variation, len(frames), expected)
        assert (sequence / "extrinsic.txt").is_file(), sequence
        assert (sequence / "intrinsic.txt").is_file(), sequence
        assert frames[0].name == "rgb_00000.jpg", sequence
        assert frames[-1].name == f"rgb_{expected - 1:05d}.jpg", sequence
        total_frames += len(frames)
        sequence_count += 1
assert (sequence_count, total_frames) == (18, 8642)
print("RGB subset OK:", sequence_count, "sequences,", total_frames, "Camera_0 frames")
if DEPTH_ROOT:
    overlay = WORK / "depth-overlay/vkitti2"
    for scene in DATA_ROOT.glob("Scene*"):
        for variation in scene.iterdir():
            if not (variation / "frames/rgb/Camera_0").is_dir():
                continue
            dest = overlay / scene.name / variation.name
            (dest / "frames").mkdir(parents=True, exist_ok=True)
            for text in variation.glob("*.txt"):
                link = dest / text.name
                if not link.exists(): link.symlink_to(text)
            link = dest / "frames/rgb"
            if not link.exists(): link.symlink_to(variation / "frames/rgb", target_is_directory=True)
            depth = Path(DEPTH_ROOT) / scene.name / variation.name / "frames/depth"
            link = dest / "frames/depth"
            if depth.exists() and not link.exists(): link.symlink_to(depth, target_is_directory=True)
    DATA_ROOT = overlay
if EVALUATE and EVAL_DEPTH:
    import numpy as np
    from PIL import Image
    assert DEPTH_ROOT and DEPTH_DATASET_VERSION, "Set the mounted depth root and dataset version"
    for scene, expected in (("Scene02", 233), ("Scene20", 837)):
        rgb_dir = DATA_ROOT / scene / "clone/frames/rgb/Camera_0"
        depth_dir = DATA_ROOT / scene / "clone/frames/depth/Camera_0"
        rgb_ids = {p.stem.removeprefix("rgb_") for p in rgb_dir.glob("rgb_*.jpg")}
        depth_ids = {p.stem.removeprefix("depth_") for p in depth_dir.glob("depth_*.png")}
        assert len(rgb_ids) == len(depth_ids) == expected and rgb_ids == depth_ids, scene
        sample = np.asarray(Image.open(depth_dir / "depth_00000.png"))
        assert sample.ndim == 2 and sample.dtype == np.uint16, scene
        print(scene, "paired depth frames:", expected, "PNG range (cm):",
              int(sample.min()), int(sample.max()))
print("data root", DATA_ROOT)


## Optional pilot: set RUN_PILOT=True

The pilot writes /kaggle/working/pilot/m3 for configs/m3.yaml, separately from
full-training outputs at /kaggle/working/checkpoints/m3. It creates a fresh
trainer on every invocation; re-running this cell can overwrite earlier evidence.
Use a fresh directory/session for a repeat. Pilot checkpoints are epoch-boundary
outputs, not mid-pilot resume points. An interrupted pilot must restart.

With the complete subset there are 4,200 training windows. PILOT_STEPS=200 and
batch 8 select 1,600 unique windows for exactly 200 optimizer updates. Selection
covers Scene01/clone (445), morning (445), overcast (445) and sunset (265), not
all training scenes. Validation uses the first eight Scene02/clone windows;
visualization then samples up to four windows across the validation sequence.
Diagnostics overrides epochs=1, selection_key=total, scheduler=none and augment=False.
It does not run test/robustness evaluation. This is integration evidence, not a
benchmark result. Expect epoch-level output, not a per-step progress bar.

### Evidence and acceptance

Preserve best.pt, last.pt, history.csv, summary.json, smoke.json,
reprojection_examples.png, depth_examples.png and loss_curves.png from PILOT.
The JSON is printed by diagnostics and again after visualization completes.

- Confirm steps=200, device=cuda and pretrained=true in smoke.json.
- Check finite train/val total, photometric and smooth losses in history.
- Check 0 <= active_fraction <= valid_fraction <= 1 and positive train active coverage.
- Confirm both encoder_max_updates values are finite and greater than zero.
- With the diagnostics evidence change, gradients.measurement is
  unscaled_before_clipping; gradients.networks.depth and .pose each report
  backward_calls=200, all_finite=true, positive finite l2_norm_min/max/mean and
  nonzero_backward_calls=200. These measure each whole network, not only the stem.
  Missing gradients means the pinned revision predates that change. The merged
  engine still rejects nonfinite gradient norms on every update; preserve its logs.
- Convert peak_gpu_memory_bytes / 1024**3 to GiB. This measures peak Torch allocation,
  not total device memory. Record GPU model and Python/Torch/Torchvision/CUDA versions.
- Inspect the target, previous/next reprojections, active masks and arbitrary-scale
  predicted depth. There is no GT depth comparison in this pilot.
- Check checkpoint epoch=0, cfg and training_commit against the pinned revision;
  preserve model/optimizer/scheduler/scaler/RNG/selection/history resume state.

A single history row cannot demonstrate decreasing loss; loss_curves.png has only
one epoch. No trajectory, benchmark metrics or automatic run.json are generated.
Record the dataset root and notebook settings and preserve deterministic warnings.
Download/persist the complete pilot directory and notebook/console outputs before
stopping the session. Never assume persistence succeeded without checking artifacts.


In [ ]:
RUN_NAME = Path(CONFIG).stem
OUT = CHECKPOINT_ROOT / RUN_NAME
PILOT = WORK / "pilot" / RUN_NAME
if RUN_PILOT:
    if not isinstance(PILOT_STEPS, int) or PILOT_STEPS <= 0:
        raise ValueError("PILOT_STEPS must be a positive integer")
    run("-m", "mvo.models.m3_selfsup_posedepth.diagnostics", "smoke", "--config", CONFIG,
        "--data-root", str(DATA_ROOT), "--out", str(PILOT), "--steps", str(PILOT_STEPS), "--device", "cuda")
    run("-m", "mvo.models.m3_selfsup_posedepth.diagnostics", "visualize",
        "--ckpt", str(PILOT / "best.pt"), "--data-root", str(DATA_ROOT), "--out", str(PILOT))
    print((PILOT / "smoke.json").read_text())
else:
    print("Pilot skipped. Set RUN_PILOT=True to request it.")


## Full run and validation-selected evaluation


In [ ]:
if RUN_FULL:
    args = ["scripts/train.py", "--model", "m3", "--config", CONFIG,
            "--data-root", str(DATA_ROOT), "--out", str(OUT), "--num-workers", str(NUM_WORKERS)]
    if (OUT / "last.pt").exists():
        if not HAS_CHECKPOINT_RESUME:
            raise RuntimeError("Existing last.pt found without the reviewed resume integration; use a fresh output directory for standalone training.")
        args += ["--resume", str(OUT / "last.pt")]
    run(*args)
else:
    print("Full training skipped. Set RUN_FULL=True to request it.")


In [ ]:
if EVALUATE:
    checkpoint = Path(EVAL_CKPT) if EVAL_CKPT else OUT / "best.pt"
    if not checkpoint.is_file():
        raise FileNotFoundError(f"Evaluation checkpoint missing: {checkpoint}; set EVAL_CKPT or restore OUT/best.pt")
    if EVAL_DEPTH:
        help_text = subprocess.check_output([PYTHON, "scripts/eval.py", "--help"], text=True)
        if "--eval-depth" not in help_text:
            raise RuntimeError("Depth evaluation requires the separately reviewed depth-evaluation integration; use EVAL_DEPTH=False for pose-only evaluation.")
    RESULTS = RESULT_ROOT / RUN_NAME / "depth_evaluation" if EVAL_DEPTH else RESULT_ROOT / RUN_NAME
    args = ["scripts/eval.py", "--model", "m3", "--ckpt", str(checkpoint),
            "--data-root", str(DATA_ROOT), "--out", str(RESULTS)]
    if EVAL_DEPTH:
        args += ["--eval-depth", "--rgb-dataset-version", RGB_DATASET_VERSION,
                 "--depth-dataset-version", DEPTH_DATASET_VERSION]
    run(*args)
    run("-m", "mvo.models.m3_selfsup_posedepth.diagnostics", "visualize",
        "--ckpt", str(checkpoint), "--data-root", str(DATA_ROOT), "--split", "test",
        "--out", str(RESULTS), *(["--with-depth"] if EVAL_DEPTH else []))
    history = checkpoint.parent / "history.csv"
    if history.is_file(): shutil.copy(history, RESULTS / "history.csv")
    print((RESULTS / "metrics.json").read_text())
    # Weather/viewpoint tests are separate output directories; depth is not required.
    if not EVAL_DEPTH:
        run("scripts/eval.py", "--model", "m3", "--ckpt", str(checkpoint),
            "--data-root", str(DATA_ROOT), "--split", "robustness", "--out", str(RESULTS / "robustness"))
